
## Folder 04 — File 04: Silver Order Items:


**Transformation rules:**

- Trim order_item_id, order_id, and product_id.
- Convert quantity to INTEGER.
- Convert unit_price and line_total to DECIMAL(12,2).
- Convert discount_rate to DECIMAL(5,4).
- Reject missing identifiers, invalid numeric values, zero or negative quantities, negative prices, and discounts outside the range 0–1.
- Deduplicate using order_item_id.
- Save valid and invalid records separately.

Business rule: **A discount rate of 0.15 means 15%. It is not 15**.

In [0]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_order_items"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_order_items"
QUARANTINE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_order_items"

print("Project configuration completed.")


# ============================================================
# 3. CREATE SILVER SCHEMA
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


# ============================================================
# 4. READ BRONZE ORDER ITEMS
# ============================================================

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(f"Bronze table not found: {BRONZE_TABLE}")

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze order item records: {bronze_df.count()}")

display(bronze_df.limit(10))


# ============================================================
# 5. VALIDATE SOURCE COLUMNS
# ============================================================

required_columns = [
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_total"
]

missing_columns = [
    column
    for column in required_columns
    if column not in bronze_df.columns
]

if missing_columns:
    raise Exception(
        f"Missing required Bronze columns: {missing_columns}"
    )

# Support either source column name.
# The known dataset uses 'discount'.

if "discount" in bronze_df.columns:
    discount_source_column = "discount"
elif "discount_rate" in bronze_df.columns:
    discount_source_column = "discount_rate"
else:
    raise Exception(
        "Neither 'discount' nor 'discount_rate' exists "
        "in the Bronze order items table."
    )

print(f"Discount source column: {discount_source_column}")


# ============================================================
# 6. CLEAN AND STANDARDIZE ORDER ITEM COLUMNS
# ============================================================

cleaned_df = (
    bronze_df

    .withColumn(
        "order_item_id",
        F.trim(F.col("order_item_id").cast("string"))
    )

    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )

    .withColumn(
        "product_id",
        F.trim(F.col("product_id").cast("string"))
    )

    .withColumn(
        "quantity",
        F.expr("try_cast(quantity AS INT)")
    )

    .withColumn(
        "unit_price",
        F.expr("try_cast(unit_price AS DECIMAL(12,2))")
    )

    .withColumn(
        "line_total",
        F.expr("try_cast(line_total AS DECIMAL(12,2))")
    )

    .withColumn(
        "discount_rate",
        F.expr(
            f"try_cast(`{discount_source_column}` AS DECIMAL(5,4))"
        )
    )
)

print("Order item standardization completed.")

display(cleaned_df.limit(10))


# ============================================================
# 7. IDENTIFY INVALID ORDER ITEM RECORDS
# ============================================================

invalid_condition = (
    F.col("order_item_id").isNull()
    | (F.col("order_item_id") == "")

    | F.col("order_id").isNull()
    | (F.col("order_id") == "")

    | F.col("product_id").isNull()
    | (F.col("product_id") == "")

    | F.col("quantity").isNull()
    | (F.col("quantity") <= 0)

    | F.col("unit_price").isNull()
    | (F.col("unit_price") < 0)

    | F.col("discount_rate").isNull()
    | (F.col("discount_rate") < 0)
    | (F.col("discount_rate") > 1)

    | F.col("line_total").isNull()
    | (F.col("line_total") < 0)
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)

    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("order_item_id").isNull()
                | (F.col("order_item_id") == ""),
                F.lit("Missing order_item_id")
            ),

            F.when(
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                F.lit("Missing order_id")
            ),

            F.when(
                F.col("product_id").isNull()
                | (F.col("product_id") == ""),
                F.lit("Missing product_id")
            ),

            F.when(
                F.col("quantity").isNull(),
                F.lit("Missing or invalid quantity")
            ),

            F.when(
                F.col("quantity") <= 0,
                F.lit("Quantity must be greater than zero")
            ),

            F.when(
                F.col("unit_price").isNull(),
                F.lit("Missing or invalid unit_price")
            ),

            F.when(
                F.col("unit_price") < 0,
                F.lit("Negative unit_price")
            ),

            F.when(
                F.col("discount_rate").isNull(),
                F.lit("Missing or invalid discount")
            ),

            F.when(
                (F.col("discount_rate") < 0)
                | (F.col("discount_rate") > 1),
                F.lit("Discount must be between 0 and 1")
            ),

            F.when(
                F.col("line_total").isNull(),
                F.lit("Missing or invalid line_total")
            ),

            F.when(
                F.col("line_total") < 0,
                F.lit("Negative line_total")
            )
        )
    )
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_df = cleaned_df.filter(~invalid_condition)

invalid_count = invalid_df.count()

print(f"Valid order items: {valid_df.count()}")
print(f"Invalid order items: {invalid_count}")

display(invalid_df.limit(20))


# ============================================================
# 8. REMOVE DUPLICATE ORDER ITEM RECORDS
# ============================================================

# Use available ingestion metadata for record precedence.

deduplication_order = []

if "_ingestion_timestamp" in valid_df.columns:
    deduplication_order.append(
        F.col("_ingestion_timestamp").desc_nulls_last()
    )

if "_ingested_at" in valid_df.columns:
    deduplication_order.append(
        F.col("_ingested_at").desc_nulls_last()
    )

if "_source_file_modification_time" in valid_df.columns:
    deduplication_order.append(
        F.col("_source_file_modification_time").desc_nulls_last()
    )

if "_source_file" in valid_df.columns:
    deduplication_order.append(
        F.col("_source_file").desc_nulls_last()
    )

# Stable business-column tie-breakers.
deduplication_order.extend([
    F.col("order_id").asc_nulls_last(),
    F.col("product_id").asc_nulls_last(),
    F.col("quantity").desc_nulls_last(),
    F.col("line_total").desc_nulls_last()
])

deduplication_window = (
    Window
    .partitionBy("order_item_id")
    .orderBy(*deduplication_order)
)

ranked_df = valid_df.withColumn(
    "_row_number",
    F.row_number().over(deduplication_window)
)

# Keep the first record for each order_item_id.

deduplicated_df = (
    ranked_df
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

# Capture duplicate records instead of silently discarding them.

duplicate_df = (
    ranked_df
    .filter(F.col("_row_number") > 1)
    .drop("_row_number")

    .withColumn(
        "_quarantine_reason",
        F.lit("Duplicate order_item_id")
    )

    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

duplicate_count = duplicate_df.count()

print(f"Valid records before deduplication: {valid_df.count()}")
print(f"Duplicate records quarantined: {duplicate_count}")
print(f"Records after deduplication: {deduplicated_df.count()}")


# ============================================================
# 9. COMBINE INVALID AND DUPLICATE RECORDS
# ============================================================

quarantine_df = invalid_df.unionByName(
    duplicate_df,
    allowMissingColumns=True
)

total_quarantine_count = quarantine_df.count()

print(f"Total quarantined records: {total_quarantine_count}")

display(
    quarantine_df
    .groupBy("_quarantine_reason")
    .count()
    .orderBy(F.desc("count"))
)


# ============================================================
# 10. SAVE QUARANTINE DELTA TABLE
# ============================================================

(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table saved: {QUARANTINE_TABLE}")


# ============================================================
# 11. SAVE CLEAN RECORDS TO SILVER DELTA TABLE
# ============================================================

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table saved: {SILVER_TABLE}")


# ============================================================
# 12. VALIDATE SILVER ORDER ITEM RECORDS
# ============================================================

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_item_groups = (
    silver_df
    .groupBy("order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_silver_records = (
    silver_df
    .filter(
        F.col("order_item_id").isNull()
        | (F.col("order_item_id") == "")

        | F.col("order_id").isNull()
        | (F.col("order_id") == "")

        | F.col("product_id").isNull()
        | (F.col("product_id") == "")

        | F.col("quantity").isNull()
        | (F.col("quantity") <= 0)

        | F.col("unit_price").isNull()
        | (F.col("unit_price") < 0)

        | F.col("discount_rate").isNull()
        | (F.col("discount_rate") < 0)
        | (F.col("discount_rate") > 1)

        | F.col("line_total").isNull()
        | (F.col("line_total") < 0)
    )
    .count()
)

print("=" * 60)
print("SILVER ORDER ITEM VALIDATION")
print("=" * 60)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate item groups: {duplicate_item_groups}")
print(f"Invalid Silver records: {invalid_silver_records}")

if duplicate_item_groups > 0 or invalid_silver_records > 0:
    raise Exception("Silver order item validation failed.")

print("Silver order item validation passed.")


# ============================================================
# 13. BRONZE-TO-SILVER RECONCILIATION
# ============================================================

bronze_distinct_ids = (
    cleaned_df
    .filter(
        F.col("order_item_id").isNotNull()
        & (F.col("order_item_id") != "")
    )
    .select("order_item_id")
    .distinct()
)

silver_distinct_ids = (
    silver_df
    .select("order_item_id")
    .distinct()
)

missing_ids_df = bronze_distinct_ids.join(
    silver_distinct_ids,
    on="order_item_id",
    how="left_anti"
)

missing_ids_count = missing_ids_df.count()

quarantined_ids = (
    quarantine_df
    .filter(
        F.col("order_item_id").isNotNull()
        & (F.col("order_item_id") != "")
    )
    .select("order_item_id")
    .distinct()
)

unexplained_missing_ids = (
    missing_ids_df
    .join(
        quarantined_ids,
        on="order_item_id",
        how="left_anti"
    )
    .count()
)

print("=" * 60)
print("BRONZE TO SILVER RECONCILIATION")
print("=" * 60)

print(f"Bronze distinct item IDs: {bronze_distinct_ids.count()}")
print(f"Silver distinct item IDs: {silver_distinct_ids.count()}")
print(f"Missing IDs from Silver: {missing_ids_count}")
print(f"Missing IDs without quarantine explanation: {unexplained_missing_ids}")

if unexplained_missing_ids > 0:
    raise Exception(
        "Some Bronze order item IDs are missing from Silver "
        "without quarantine explanation."
    )

print("Bronze-to-Silver reconciliation passed.")


# ============================================================
# 14. CHECK REFERENTIAL INTEGRITY
# ============================================================

# These checks are informational.
# Do not automatically quarantine otherwise valid order items
# just because a parent order or product is missing.

ORDERS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
PRODUCTS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_products"

if spark.catalog.tableExists(ORDERS_TABLE):

    orders_df = spark.table(ORDERS_TABLE).select("order_id").distinct()

    items_without_orders = (
        silver_df
        .join(orders_df, on="order_id", how="left_anti")
        .count()
    )

    print(f"Order items without matching Silver orders: {items_without_orders}")

else:
    print("Silver Orders table not found. Order reference check skipped.")


if spark.catalog.tableExists(PRODUCTS_TABLE):

    products_df = (
        spark.table(PRODUCTS_TABLE)
        .select("product_id")
        .distinct()
    )

    items_without_products = (
        silver_df
        .join(products_df, on="product_id", how="left_anti")
        .count()
    )

    print(f"Order items without matching Silver products: {items_without_products}")

else:
    print("Silver Products table not found. Product reference check skipped.")


# ============================================================
# 15. CALCULATE ORDER ITEM METRICS
# ============================================================

display(
    silver_df
    .agg(
        F.count("*").alias("total_order_items"),
        F.sum("quantity").alias("total_units_sold"),
        F.sum("line_total").alias("total_line_sales"),
        F.avg("discount_rate").alias("average_discount_rate")
    )
)


# ============================================================
# 16. DISPLAY FINAL SILVER ORDER ITEM DATA
# ============================================================

display(
    silver_df
    .orderBy("order_item_id")
    .limit(20)
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

bronze_count = bronze_df.count()
silver_count = total_silver_records

print("=" * 60)
print("ORDER ITEM SILVER TRANSFORMATION COMPLETED")
print("=" * 60)

print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Silver table     : {SILVER_TABLE}")
print(f"Quarantine table : {QUARANTINE_TABLE}")

print(f"Bronze records   : {bronze_count}")
print(f"Silver records   : {silver_count}")
print(f"Invalid records  : {invalid_count}")
print(f"Duplicate records: {duplicate_count}")
print(f"Total quarantined: {total_quarantine_count}")

print("=" * 60)



## Quarantined order:

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("PySpark functions imported successfully.")

quarantine_df = spark.table(
    "workspace.ecommerce_silver.quarantine_order_items"
)

display(
    quarantine_df
    .filter(
        F.col("order_id").isin(
            "ORD0000778",
            "ORD0008778"
        )
    )
    .select(
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount_rate",
        "line_total",
        "_quarantine_reason"
    )
)

print("Total quarantined order items:", quarantine_df.count())

In [0]:
from pyspark.sql import functions as F

quarantine_df = spark.table(
    "workspace.ecommerce_silver.quarantine_order_items"
)

silver_df = spark.table(
    "workspace.ecommerce_silver.silver_order_items"
)

print("QUARANTINE VALIDATION")
print("Quarantined records:", quarantine_df.count())

display(
    quarantine_df
    .groupBy("_quarantine_reason")
    .count()
)

print("SILVER VALIDATION")
print("Silver records:", silver_df.count())

display(
    quarantine_df.filter(
        F.col("order_item_id").isin(
            "ITEM0000778",
            "ITEM0008778"
        )
    )
)

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT_IF(order_item_id IS NULL OR TRIM(order_item_id) = '') AS missing_order_item_id,
    COUNT_IF(order_id IS NULL OR TRIM(order_id) = '') AS missing_order_id,
    COUNT_IF(product_id IS NULL OR TRIM(product_id) = '') AS missing_product_id,
    COUNT_IF(quantity IS NULL) AS missing_quantity,
    COUNT_IF(unit_price IS NULL) AS missing_unit_price,
    COUNT_IF(discount_rate IS NULL) AS missing_discount_rate,
    COUNT_IF(line_total IS NULL) AS missing_line_total
FROM workspace.ecommerce_silver.silver_order_items;

In [0]:
%sql
SELECT
    _quarantine_reason,
    COUNT(*) AS record_count
FROM workspace.ecommerce_silver.quarantine_order_items
GROUP BY _quarantine_reason
ORDER BY record_count DESC;